# Tagging a Sequence: Hidden Markov Models and Viterbi

**Computational Text Analysis Pack · Module 16 — Sequence Labelling**

*Technique:* HMM tagging from scratch; emission and transition probabilities; Viterbi

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/06-supervised-classification/16a_hmm_pos_tagging.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Notebook 15b built a corpus query language and then admitted its weakness: the
part-of-speech tags came from a handful of English suffix rules, so every
grammatical query inherited their errors.

This notebook builds a real tagger. The classic method is a **hidden Markov
model**: the words are what you observe, the tags are hidden, and you recover the
most likely tag sequence with the **Viterbi** algorithm.

The key idea is that a tag depends on its *neighbours*. A document classifier
labels each item independently; a sequence labeller cannot, because "the" is
followed by a noun far more often than by a verb, and that constraint is most of
the signal.

## The tooling

**NLTK** ships HMM and perceptron taggers and is the standard teaching
implementation. **spaCy** and **Stanza** ship neural taggers that are far more
accurate on well-resourced languages. **CRFsuite** / `sklearn-crfsuite`
implements conditional random fields, which is what most DH sequence-labelling
projects used before neural models, and **MALLET** does the same on the JVM.

We write Viterbi out because it is twenty lines, and because seeing the
transition matrix is the only way to understand what a tagger actually knows.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import nltk

# NLTK's tagger gives us training labels. In a real project this is where your
# hand-annotated data goes — and hand-annotation is what module 17 is about.
sentences = []
for passage in df_colonial["text"].head(400):
    for raw in str(passage).split("."):
        toks = tokenise(raw)
        if 5 <= len(toks) <= 30:
            sentences.append(toks)
print(f"{len(sentences)} sentences for training and testing")

tagged = [nltk.pos_tag(s) for s in sentences[:1200]]
SIMPLE = {"NN": "NOUN", "NNS": "NOUN", "NNP": "NOUN", "NNPS": "NOUN",
          "VB": "VERB", "VBD": "VERB", "VBG": "VERB", "VBN": "VERB",
          "VBP": "VERB", "VBZ": "VERB", "JJ": "ADJ", "JJR": "ADJ", "JJS": "ADJ",
          "RB": "ADV", "RBR": "ADV", "RBS": "ADV", "DT": "DET", "IN": "ADP",
          "PRP": "PRON", "PRP$": "PRON", "CC": "CONJ", "CD": "NUM"}
data = [[(w, SIMPLE.get(t, "X")) for w, t in s] for s in tagged]
split = int(len(data) * 0.8)
train, test = data[:split], data[split:]
print(f"{len(train)} training / {len(test)} test sentences")
print("\nexample:", train[0][:8])

## The two probability tables

An HMM is nothing but two counts turned into probabilities.

- **Emission** — given a tag, how likely is this word? `P(word | tag)`
- **Transition** — given the previous tag, how likely is this tag? `P(tag | prev)`

That second table is the part a document classifier does not have.

In [ ]:
emission = defaultdict(Counter)
transition = defaultdict(Counter)
tag_counts = Counter()

for sent in train:
    prev = "<START>"
    for word, tag in sent:
        emission[tag][word] += 1
        transition[prev][tag] += 1
        tag_counts[tag] += 1
        prev = tag
    transition[prev]["<END>"] += 1

TAGS = sorted(tag_counts)
vocab = {w for tag in emission for w in emission[tag]}
print(f"{len(TAGS)} tags, {len(vocab):,} word types\n")

print("What follows a determiner?")
det = transition["DET"]
total = sum(det.values())
for tag, n in det.most_common(6):
    print(f"   DET -> {tag:<7}{n / total:>8.1%}")
print("\nThat asymmetry is the constraint the tagger exploits.")

In [ ]:
def p_emit(word, tag, alpha=0.1):
    return (emission[tag][word] + alpha) / (tag_counts[tag] + alpha * len(vocab))

def p_trans(prev, tag, alpha=0.1):
    row = transition[prev]
    return (row[tag] + alpha) / (sum(row.values()) + alpha * len(TAGS))

def viterbi(words):
    V = [{t: (np.log(p_trans("<START>", t)) + np.log(p_emit(words[0], t)), None)
          for t in TAGS}]
    for i in range(1, len(words)):
        layer = {}
        for t in TAGS:
            best_prev, best_score = None, -np.inf
            for pt in TAGS:
                s = V[i - 1][pt][0] + np.log(p_trans(pt, t))
                if s > best_score:
                    best_score, best_prev = s, pt
            layer[t] = (best_score + np.log(p_emit(words[i], t)), best_prev)
        V.append(layer)
    best_tag = max(TAGS, key=lambda t: V[-1][t][0])
    out = [best_tag]
    for i in range(len(words) - 1, 0, -1):
        best_tag = V[i][best_tag][1]
        out.append(best_tag)
    return out[::-1]

demo = [w for w, _ in test[0]]
print("sentence:", " ".join(demo))
print("predicted:", " ".join(f"{w}/{t}" for w, t in zip(demo, viterbi(demo))))
print("gold     :", " ".join(f"{w}/{t}" for w, t in test[0]))

## How good is it, and where does the sequence actually help?

The comparison that matters is against a tagger that ignores context entirely —
one that gives every word its most frequent tag. The difference between the two
is precisely what the transition table bought.

In [ ]:
most_frequent = {}
for tag in emission:
    for w, n in emission[tag].items():
        if w not in most_frequent or n > emission[most_frequent[w]][w]:
            most_frequent[w] = tag
fallback = tag_counts.most_common(1)[0][0]

hmm_ok = base_ok = total_tok = 0
for sent in test[:120]:
    words = [w for w, _ in sent]
    gold = [t for _, t in sent]
    pred = viterbi(words)
    for w, g, p in zip(words, gold, pred):
        total_tok += 1
        hmm_ok += (p == g)
        base_ok += (most_frequent.get(w, fallback) == g)

print(f"tokens evaluated            : {total_tok:,}")
print(f"most-frequent-tag baseline  : {base_ok / total_tok:.3f}")
print(f"HMM with Viterbi            : {hmm_ok / total_tok:.3f}")
print()
print("The gap is the value of looking at neighbours. It is usually modest in")
print("English and much larger in languages with freer word order — which is")
print("also where the training data does not exist.")

In [ ]:
# Where the sequence model earns its keep: genuinely ambiguous words.
ambiguous = [(w, {t: emission[t][w] for t in TAGS if emission[t][w]})
             for w in list(vocab)[:4000]]
ambiguous = [(w, d) for w, d in ambiguous if len(d) > 1]
ambiguous.sort(key=lambda kv: -sum(kv[1].values()))
print("Most frequent words carrying more than one tag:\n")
for w, dist in ambiguous[:10]:
    total = sum(dist.values())
    spread = "  ".join(f"{t} {n/total:.0%}" for t, n in
                       sorted(dist.items(), key=lambda kv: -kv[1])[:3])
    print(f"   {w:<14}{spread}")
print()
print("For these words the context is the only evidence. A bag-of-words model")
print("cannot use it; that is the structural difference, not a tuning detail.")

### 🔧 Your turn 1

Increase the training set from 1200 sentences to the full set and re-run. Does the gap over the baseline widen or narrow?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The baseline improves faster than the HMM at first, because most of the easy win
is simply knowing each word's usual tag, and that needs only counts. The HMM's
advantage concentrates on unknown words and ambiguous ones, so it grows most
where the vocabulary is open — which in practice means proper nouns and
borrowings. This is also why CRFs displaced HMMs: a CRF can condition on
arbitrary features of the word (suffix, capitalisation, shape) rather than only
its identity, so it handles unknown words far better without needing more data.

</details>